# EDA: corpus, filter, and topic model

One question this notebook answers end to end: **how much of the raw arXiv snapshot survives each stage of the pipeline, and what does the surviving corpus look like?**

Reads only already-computed artifacts under `data/interim/` and `configs/` (produced by `atlas embed` / `atlas topics` / `atlas register-topic` / `atlas story`) - this notebook does not refit anything. See `docs/decisions/0003-medical-imaging-filter-rule.md`, `0004-embedding-and-topic-model-choice.md`, and `0005-config-driven-filter-rule.md` for the decisions behind the numbers below.

In [ ]:
from pathlib import Path

import pandas as pd

from imaging_ai_atlas.data.filter import apply_filter, load_filter_config
from imaging_ai_atlas.data.quality import compute_quality_report

papers = pd.read_parquet("../data/interim/papers.parquet")
print(f"{len(papers):,} rows in the raw ingested corpus")

## 1. Data quality (raw corpus)

`compute_quality_report` is the same function behind `reports/data_quality.md` - reproduced here rather than duplicated by hand, so this notebook can't silently drift from that report.

In [ ]:
report = compute_quality_report(papers)
print(f"Date range: {report.date_min} to {report.date_max}")
print(f"Pre-launch (backdated) records: {report.pre_launch_count}")
print(f"Degenerate short abstracts: {report.short_abstract_count}")
print(f"Abstracts with LaTeX markup: {report.latex_leakage_count:,} ({report.latex_leakage_count / report.row_count:.1%})")

## 2. Filter stage

The filter rule is config-driven (ADR 0005) - swap `filter_config_path` for a different YAML to point this whole notebook at a different research-topic domain, no code change.

In [ ]:
filter_config_path = Path("../configs/filter_medical_imaging.yaml")
filter_config = load_filter_config(filter_config_path)
in_filter = apply_filter(papers, filter_config)
filtered = papers.loc[in_filter].reset_index(drop=True)

print(f"Filter domain: {filter_config.name!r}")
print(f"Accepted: {in_filter.sum():,} of {len(papers):,} ({in_filter.mean():.2%})")

In [ ]:
# Category distribution within the accepted set - eess.IV and cs.CV dominate by design
# (ADR 0001/0003); worth seeing the actual split rather than assuming.
filtered["categories"].explode().value_counts().head(10)

## 3. Precision/recall validation sample

`filter_labeling_sample_validation.csv` is the human-labeled sample behind ADR 0003's precision/recall numbers (0.895 precision; recall unresolved, see the ADR's discussion of the 104x class-imbalance risk).

In [ ]:
validation = pd.read_csv(
    "../data/interim/filter_labeling_sample_validation.csv", sep=";", encoding="utf-8-sig"
)
validation.groupby("in_filter")["label"].value_counts()

## 4. Topic model summary

`topic_assignments.parquet` is the registered nomic@64 model's per-document topic assignment (ADR 0004); `topic_labeling_sample.csv` is the human coherence check on the 30 largest topics (all 30 marked coherent).

In [ ]:
assignments = pd.read_parquet("../data/interim/topic_assignments.parquet")
n_topics = assignments.loc[assignments["topic_id"] != -1, "topic_id"].nunique()
outlier_rate = (assignments["topic_id"] == -1).mean()

print(f"{n_topics} topics found over {len(assignments):,} documents")
print(f"Outlier rate (before reduction): {outlier_rate:.1%}")

In [ ]:
labels = pd.read_csv("../data/interim/topic_labeling_sample.csv")
print(f"{(labels['coherent'].str.strip().str.lower() == 'yes').sum()} / {len(labels)} "
      "of the largest topics marked coherent by human review")
labels[["topic_id", "size", "suggested_label"]].head(10)

## 5. Topic map and topic-share-over-time charts

The 2D topic map and topic-share-over-time dynamics chart, built by `atlas story` (`src/imaging_ai_atlas/story/`). Rendered here for a quick visual check; the authoritative copies live in `reports/figures/`.

![Topic map](../reports/figures/topic_map.png)

![Topic dynamics](../reports/figures/topic_dynamics.png)